# MovieLens-20M: leader data infrastructure

This notebook is intentionally limited to loading, quality checks, shared preprocessing tables, dataset scale, the global rating baseline, and runtime feasibility. It does not answer the contributor questions assigned to Persons B, C, or D.

In [ ]:
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from load_data import load_movielens, summarize_frames
from preprocess import build_shared_tables, prepare_genome_scores, prepare_genome_tags
from metrics import data_quality_report, rating_distribution, runtime_feasibility
from plotting import plot_rating_distribution, save_figure

In [ ]:
data_dir = Path(os.environ.get('MOVIELENS_DATA_DIR', PROJECT_ROOT / 'data' / 'ml-20m'))
frames = load_movielens(data_dir)
ratings_raw = frames['ratings']
movies_raw = frames['movies']
tags_raw = frames.get('tags', pd.DataFrame(columns=['userId', 'movieId', 'tag', 'timestamp']))
print('Loaded data directory:', frames['_data_dir'])
display(summarize_frames(frames))

In [ ]:
tables = build_shared_tables(ratings_raw, movies_raw, tags_raw)
ratings = tables['ratings']
movies = tables['movies']
tags = tables['tags']
movie_stats = tables['movie_stats']
user_stats = tables['user_stats']
exploded_genres = tables['exploded_genres']
rating_genre_df = tables['rating_genre_df']
movie_tag_stats = tables['movie_tag_stats']

genome_scores = frames.get('genome-scores', pd.DataFrame())
genome_tags = frames.get('genome-tags', pd.DataFrame())
genome_scores = prepare_genome_scores(genome_scores) if not genome_scores.empty else None
genome_tags = prepare_genome_tags(genome_tags) if not genome_tags.empty else None

quality = data_quality_report(frames, ratings, movies, tags, genome_scores, genome_tags)
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'summary_tables'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
quality['file_summary'].to_csv(OUTPUT_DIR / 'file_summary.csv', index=False)
quality['key_integrity'].to_csv(OUTPUT_DIR / 'key_integrity.csv', index=False)
display(quality['file_summary'])
display(quality['key_integrity'])

## Shared tables and dataset scale

These tables are the handoff contract. The leader prepares their columns and join keys; Persons B, C, and D perform the analyses on top of them.

In [ ]:
scale = pd.DataFrame({
    'metric': ['users', 'movies', 'ratings', 'tag applications', 'genome tags', 'rating start', 'rating end', 'tag start', 'tag end'],
    'value': [
        ratings['userId'].nunique(),
        movies['movieId'].nunique(),
        len(ratings),
        len(tags),
        genome_tags['tagId'].nunique() if genome_tags is not None else np.nan,
        ratings['rating_datetime'].min(),
        ratings['rating_datetime'].max(),
        tags['tag_datetime'].min() if not tags.empty else pd.NaT,
        tags['tag_datetime'].max() if not tags.empty else pd.NaT,
    ],
})
display(scale)
display(pd.DataFrame({name: [len(frame)] for name, frame in tables.items() if isinstance(frame, pd.DataFrame)}, index=['rows']).T)
display(movies[['movieId', 'title', 'release_year', 'release_year_parse_ok']].head())

## Leader-owned rating baseline

This is the dataset-level rating distribution required for the shared project context. User behavior, genre, and tag findings are deliberately left to the assigned contributors.

In [ ]:
rating_counts, rating_stats = rating_distribution(ratings)
rating_counts.to_csv(OUTPUT_DIR / 'rating_counts.csv', index=False)
rating_stats.to_csv(OUTPUT_DIR / 'rating_summary.csv', index=False)
display(rating_stats)
display(rating_counts)
rating_fig = plot_rating_distribution(rating_counts)
save_figure(rating_fig, PROJECT_ROOT / 'figures' / 'rating_distribution.png')
plt.show()

## Runtime feasibility

Runtime is not inferred from title length. This records whether a reliable runtime analysis is possible from the supplied files and identifiers.

In [ ]:
runtime_result = runtime_feasibility(frames.get('links'), movies)
pd.DataFrame([runtime_result]).to_csv(OUTPUT_DIR / 'runtime_feasibility.csv', index=False)
display(pd.Series(runtime_result, name='value').to_frame())

## Contributor boundary

Do not add B/C/D findings here. Persons B, C, and D should implement their own questions, tables, plots, interpretations, limitations, and handoff files in notebooks/01-03.